# 2.字符串提示拼接

知识点讲解：PromptTemplate 字符串拼接技巧

1. 拼接的应用场景
   - 在实际应用中，提示词可能由多个部分组成（通用前缀 + 具体任务 + 格式要求等）
   - 通过拼接可以实现提示词的模块化管理和复用，适合将固定文本和模板化文本组合使用
   - 可以提前定义通用的提示词片段，需要时动态组合，注意拼接顺序确保最终语义连贯

2. + 操作符背后的 __add__ 重载
   - LangChain 支持使用 + 操作符直接拼接 PromptTemplate 对象，也可混合拼接普通字符串
   - 这源于 Prompt 组件重写了 __add__ 魔术方法，因此「几乎所有 Prompt 组件都可以用 + 组装拼接」是框架层面的统一约定，而非 PromptTemplate 的特例
   - = 是左操作数的方法，左侧必须是 PromptTemplate 对象，右侧可以是 PromptTemplate 或普通 str（框架内部做类型归一）
   - 对比 Runnable 组件重写的是 __or__ / __ror__ 来支撑 | 管道，两者是不同层面的运算符重载，不要混淆（| 的用法见 4-LCEL表达式与Runnable可运行协议）

3. 拼接后变量自动合并
   - 拼接后的模板会自动合并所有部分的输入变量到同一个 input_variables 集合
   - 拼接发生在「构建期」，运行期没有额外开销；产物仍是单一的 PromptTemplate，而不是保存多个片段的容器
   - invoke 时需要同时提供所有变量的值（本例需传 subject 与 language）

4. 典型输出示例与观察点
   - 直接打印拼接后的模板对象，可见变量已被自动合并到 input_variables，且三段文本已合并成一个 template 字符串：
       input_variables=['language', 'subject']
       template='请讲一个关于{subject}的冷笑话，让我开心下
使用{language}语言。'
   - 观察点 1：input_variables 是合并后的集合，按字母排序，与书写顺序无关
   - 观察点 2：拼接产物是一个「单一的 PromptTemplate」，运行期零额外开销
   - 调用 invoke(...).to_string() 后得到的格式化输出：
       请讲一个关于程序员的冷笑话，让我开心下
       使用中文语言。

5. 与 ChatPromptTemplate 拼接的差异
   - 本例是「字符串提示拼接」：合并的是 template 文本，结果是一个 PromptTemplate
   - ChatPromptTemplate 的 + 合并的是 messages 列表，结果是一个 ChatPromptTemplate（详见 3.消息提示模板拼接.py）

6. 使用建议
   - 适合将固定文本和模板化文本组合使用，可提前定义通用提示词片段，需要时动态组合
   - 注意拼接顺序，确保最终生成的提示词语义连贯
   - 拼接得到的单个模板对象仍可用 partial / format / invoke 进一步处理


In [ ]:
from langchain_core.prompts import PromptTemplate
prompt = (
        PromptTemplate.from_template("请讲一个关于{subject}的冷笑话")
        + ",让我开心下" +
        "\n使用{language}语言"
)
print(prompt.invoke({"subject": "程序员", "language": "中文"}).to_string())